In [59]:
import wave, math, struct
import tensorflow as tf
import numpy as np
from keras.models import Sequential
from keras.layers import Dense, LSTM, GRU, Activation

In [60]:
# prepare a dumy data for simulating musical notes
notes_freqs={
    'A':440.0, 'B':493.88, 'C':261.63, 'D':293.66, 'E':393.63, 'F':349.23, 'G':392.0
}

In [61]:
notes_freqs

{'A': 440.0,
 'B': 493.88,
 'C': 261.63,
 'D': 293.66,
 'E': 393.63,
 'F': 349.23,
 'G': 392.0}

In [62]:
notes = list(notes_freqs.keys())

In [63]:
notes

['A', 'B', 'C', 'D', 'E', 'F', 'G']

In [64]:
note_to_int={note  : i for i, note in enumerate(notes)}

In [65]:
int_to_note = {i:note for i, note in enumerate(notes)}
int_to_note

{0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E', 5: 'F', 6: 'G'}

In [66]:
raw_music_data = [notes[np.random.randint(0,7)] for i in range(1000)]

In [67]:
raw_music_data

['D',
 'F',
 'C',
 'C',
 'B',
 'E',
 'F',
 'F',
 'B',
 'D',
 'E',
 'B',
 'A',
 'B',
 'C',
 'D',
 'D',
 'F',
 'C',
 'A',
 'C',
 'E',
 'E',
 'G',
 'A',
 'E',
 'F',
 'C',
 'C',
 'F',
 'D',
 'G',
 'B',
 'E',
 'B',
 'A',
 'F',
 'B',
 'D',
 'A',
 'B',
 'D',
 'B',
 'C',
 'B',
 'F',
 'G',
 'A',
 'B',
 'C',
 'F',
 'D',
 'G',
 'A',
 'E',
 'B',
 'D',
 'A',
 'E',
 'A',
 'G',
 'A',
 'A',
 'C',
 'B',
 'C',
 'B',
 'A',
 'D',
 'E',
 'F',
 'D',
 'C',
 'D',
 'E',
 'G',
 'F',
 'A',
 'D',
 'C',
 'A',
 'A',
 'E',
 'G',
 'B',
 'F',
 'D',
 'G',
 'A',
 'C',
 'D',
 'B',
 'E',
 'A',
 'B',
 'B',
 'E',
 'C',
 'D',
 'D',
 'D',
 'A',
 'G',
 'F',
 'B',
 'D',
 'F',
 'E',
 'A',
 'C',
 'B',
 'C',
 'D',
 'F',
 'B',
 'E',
 'A',
 'E',
 'B',
 'A',
 'B',
 'D',
 'B',
 'D',
 'G',
 'B',
 'C',
 'F',
 'D',
 'B',
 'G',
 'E',
 'G',
 'E',
 'D',
 'G',
 'G',
 'D',
 'C',
 'A',
 'A',
 'B',
 'G',
 'G',
 'E',
 'C',
 'B',
 'D',
 'G',
 'F',
 'G',
 'C',
 'G',
 'D',
 'E',
 'C',
 'C',
 'B',
 'G',
 'E',
 'B',
 'C',
 'G',
 'C',
 'C',
 'D',
 'F'

In [68]:
seq_length=3
network_input=[]
network_output= []

for i in range(len(raw_music_data) - seq_length):
    seq_in = raw_music_data[i: i+seq_length]
    seq_out= raw_music_data[i+seq_length]
    network_input.append([note_to_int[char] for char in seq_in])
    network_output.append(note_to_int[seq_out])
    print(seq_in, '---->',seq_out)

['D', 'F', 'C'] ----> C
['F', 'C', 'C'] ----> B
['C', 'C', 'B'] ----> E
['C', 'B', 'E'] ----> F
['B', 'E', 'F'] ----> F
['E', 'F', 'F'] ----> B
['F', 'F', 'B'] ----> D
['F', 'B', 'D'] ----> E
['B', 'D', 'E'] ----> B
['D', 'E', 'B'] ----> A
['E', 'B', 'A'] ----> B
['B', 'A', 'B'] ----> C
['A', 'B', 'C'] ----> D
['B', 'C', 'D'] ----> D
['C', 'D', 'D'] ----> F
['D', 'D', 'F'] ----> C
['D', 'F', 'C'] ----> A
['F', 'C', 'A'] ----> C
['C', 'A', 'C'] ----> E
['A', 'C', 'E'] ----> E
['C', 'E', 'E'] ----> G
['E', 'E', 'G'] ----> A
['E', 'G', 'A'] ----> E
['G', 'A', 'E'] ----> F
['A', 'E', 'F'] ----> C
['E', 'F', 'C'] ----> C
['F', 'C', 'C'] ----> F
['C', 'C', 'F'] ----> D
['C', 'F', 'D'] ----> G
['F', 'D', 'G'] ----> B
['D', 'G', 'B'] ----> E
['G', 'B', 'E'] ----> B
['B', 'E', 'B'] ----> A
['E', 'B', 'A'] ----> F
['B', 'A', 'F'] ----> B
['A', 'F', 'B'] ----> D
['F', 'B', 'D'] ----> A
['B', 'D', 'A'] ----> B
['D', 'A', 'B'] ----> D
['A', 'B', 'D'] ----> B
['B', 'D', 'B'] ----> C
['D', 'B', 'C'] 

In [69]:
n_pattern = len(network_input)
n_pattern

997